# BlindSpot v0.4: new possession features, held-out comparison

Run the cells top to bottom (**Runtime → Run all** works). A normal **CPU** runtime is enough; no GPU needed.

1. Installs pinned packages.
2. Asks you to upload `BlindSpot_v04_project.zip` (built locally in `data/drive-transfer/`). Its SHA256 is checked.
3. Downloads the four tactical tracking files (~220 MB, no video) from the shared Drive copy and checks their SHA256.
4. Runs the unit tests.
5. Runs every v0.4 experiment (about 15–25 min) and shows the results table.
6. Exports a candidate `CoreOrigin.csv` for the group you choose and validates it.
7. Downloads a ZIP with `EXPERIMENTS_v04.md` and the candidate. Send the results table back.

Scores reuse the same inspected public five minutes. Trying and selecting feature groups on this sample makes this exploratory development, not independent validation. New features use offline raw context that may extend across a full half; the 300-frame embargo excludes training labels, not every shared tracking observation. No private-match score is known.


In [ ]:
%pip -q install numpy==2.2.6 pandas==2.2.3 scipy==1.17.1 scikit-learn==1.8.0 joblib==1.5.3 threadpoolctl==3.6.0 gdown==5.2.0


## 1. Upload and verify the source bundle

A file picker appears. Choose `BlindSpot_v04_project.zip`. Skipped if already uploaded in this runtime.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

BUNDLE_SHA256 = '445d2cd2c0963a4feddfb42c09cab49e3c4cc7d0e11ca6fce12aacf951b4d897'
PROJECT = Path('/content/blindspot-v04')
package = Path('/content/BlindSpot_v04_project.zip')

def sha256(path):
    value = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()

if not package.exists() or sha256(package) != BUNDLE_SHA256:
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    if Path(name).resolve() != package.resolve():
        shutil.move(name, package)
digest = sha256(package)
assert digest == BUNDLE_SHA256, f'Bundle checksum mismatch: got {digest}. Rebuild with scripts/build_v04_colab.py and use the matching notebook.'
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(package) as archive:
    for entry in archive.infolist():
        if not (PROJECT / entry.filename).resolve().is_relative_to(PROJECT.resolve()):
            raise ValueError('Unsafe bundle path: ' + entry.filename)
    archive.extractall(PROJECT)
os.chdir(PROJECT)
os.environ['LOKY_MAX_CPU_COUNT'] = '2'
print('Verified source bundle', digest, '->', PROJECT)


## 2. Download and verify the tactical tracking inputs


In [ ]:
import gdown
ASSETS = {'BAR-ATH_tactical_atd.txt': '1JEPmnLYax4dnVh1ZlInJAH5calmS1tIz', 'BAR-ATH_tactical_atd.xml': '1WBfy2sOwmJ2cX2w9XJq6y4VXo_kxKHjX', 'BAR-ATH_tactical_homography.csv': '1tuq2EGFmfvnXl8u5dU7JLoN3eNbrWwDJ', 'BAR-ATH_tactical_smart_tagging.xml': '1SvfntIRtFh6i7KAK1gVJk1NQbS_lus2K'}
EXPECTED = {'BAR-ATH_tactical_atd.txt': 'f130e350d2bfdf734b7e32f072931fccc6d015c1e837d082064875fe0a700431', 'BAR-ATH_tactical_atd.xml': '327e35e32e46a647389b88f883d4f9a2ab931e3e6148aef2fcf8f22b1e93c43a', 'BAR-ATH_tactical_homography.csv': '421c81abe16f0912dc45c2541f9bf19dd5660066397a3dcc043038a1b61c28c3', 'BAR-ATH_tactical_smart_tagging.xml': '0594d35e3df01b9c5002372d8615e2e8ef5379ac8f4e84c5ef7a53f4304c9019'}
data_dir = PROJECT / 'data/tactical'
data_dir.mkdir(parents=True, exist_ok=True)
for name, file_id in ASSETS.items():
    path = data_dir / name
    if not path.exists() or sha256(path) != EXPECTED[name]:
        if gdown.download(id=file_id, output=str(path), quiet=False) is None:
            raise RuntimeError('Download failed (sharing or Drive quota): ' + name)
    assert sha256(path) == EXPECTED[name], 'Dataset checksum mismatch: ' + name
    print('Verified', name, path.stat().st_size, 'bytes')


## 3. Unit tests


In [ ]:
result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests'], capture_output=True, text=True)
print(result.stdout[-3000:]); print(result.stderr[-3000:])
result.check_returncode()


## 4. Run all experiments

Prints one line per feature group as it finishes. The first step reads the raw tracking once (a few minutes).
`baseline` must print whole-run **0.4457** and fixed-time **0.6200**; if not, stop and report it.


In [ ]:
def stream(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)

stream([sys.executable, '-u', 'scripts/run_v04_experiments.py', '--note', 'Colab CPU run'])
stream([sys.executable, 'scripts/report_v04_results.py'])


## 5. Results


In [ ]:
from IPython.display import Markdown, display
text = (PROJECT / 'EXPERIMENTS_v04.md').read_text()
display(Markdown(text[text.index('## Results'):]))


## 6. Export a candidate

The conservative development rule below retains `baseline` unless a group improves spell F1 by more than 0.06 in **both** protocols. This is a selection heuristic, not a statistical significance threshold. Selection still reuses the public sample.
Options: `baseline`, `contacts`, `asym`, `gap_out`, `ballmotion`, `shape`, `all`, `contacts_team_only`.


In [ ]:
scores = json.loads((PROJECT / 'data/experiments/blindspot-v04/scores.json').read_text())
baseline = scores['baseline']
eligible = [name for name, score in scores.items() if name != 'baseline' and all(
    score[protocol]['spell'] - baseline[protocol]['spell'] > .06
    for protocol in ('whole_run', 'fixed_time'))]
GROUP = max(eligible, key=lambda name: min(scores[name][p]['spell'] - baseline[p]['spell']
    for p in ('whole_run', 'fixed_time'))) if eligible else 'baseline'
print('Provisional export group:', GROUP, '| same-sample development selection')

stream([sys.executable, '-u', 'scripts/run_v04_experiments.py', '--candidate', GROUP])
candidate = PROJECT / f'data/experiments/blindspot-v04/{GROUP}/candidate/CoreOrigin.csv'
stream([sys.executable, 'evaluation/validate_submission.py', str(candidate)])
print('TRAINING FIT ONLY: the exported model used all public labels. The next score is not held-out validation.')
stream([sys.executable, 'evaluation/run_evaluation.py', '--gt', 'public', '--results', str(candidate.parent),
        '--out', str(PROJECT / 'data/experiments/blindspot-v04/training_fit_only'), '--no-detail'])
stream([sys.executable, 'scripts/report_v04_results.py'])


## 7. Download results

The ZIP holds the experiment log, fold predictions/scores, provenance, report and candidate. Keep these match-derived outputs within the permitted sharing scope.


In [ ]:
from google.colab import files
bundle_dir = Path('/content/BlindSpot_v04_results')
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir()
shutil.copy(PROJECT / 'EXPERIMENTS_v04.md', bundle_dir)
shutil.copytree(PROJECT / 'data/experiments/blindspot-v04', bundle_dir / 'experiment')
files.download(shutil.make_archive('/content/BlindSpot_v04_results', 'zip', bundle_dir))
